In [2]:
import cv2
import numpy as np
cap = cv2.VideoCapture(0) #videocapture
if not cap.isOpened():
    print("웹캡을 열 수 업습니다.")
    exit()
while True:
    ret, frame = cap.read() #cap 읽어와서 언패킹 ret 정상적으로 읽었는지 (True/False), frame 읽어온 이미지 프레임(NumPy 배열), 보통 shape (height, width, 3) (BGR)
    if not ret:
        print("프레임을 가져 올수 없습니다.")
        break
    flip_fram =cv2.flip(frame,1) #거울
    height,width,_ = frame.shape #(세로, 가로, 채널) = (height, width, channels) (OpenCV에서 cols=가로, rows=세로 / ML feature는 데이터 열 의미)
    center_x,center_y = width//2,height//2 #반때기 중심
    roi = flip_fram[center_y -150 :center_y+150,center_x-150:center_x+150] #NumPy 슬라이싱 문법 a[start:end]
#enter_y -150 부터 :center_y+150 까지 자르기
    cv2.rectangle(flip_fram,(center_x - 150,center_y - 150),(center_x + 150,center_y + 150),(0,255,0),2)
    #center_x - 150 : 중심에서 왼쪽으로 150픽셀
    #center_y - 150 : 중심에서 위로 150픽셀
    #사각형의 왼쪽 위
    #오른쪽 아래 꼭짓점

    cv2.imshow('Webcam',flip_fram)#화면 좌우반전 나오게 했음
    #화면 캡쳐를 위한 키 값 받기
    key = cv2.waitKey(1) & 0xFF # 1ms기다리며  반환값에서 하위 8비트만 쓰겠다는 처리
    if key == ord('c'or 'C'): #c capture의 약자
        gray_img = cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY)#그레이 컬러
        gray_img=np.flip(gray_img,1) #cv2.flip이랑 비슷
        cv2.imwrite('gray_image.png',gray_img)#이미지 저장
        gaussian_blur = cv2.GaussianBlur(gray_img,(5,5),3)#gaussianblur 5*5 커널 크기(불러 범위) 3 : sigmaX(가우시안 표준편차) “선굵기” 아님

#이진화
        _,otsu_thread = cv2.threshold(gaussian_blur,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        #THRESH_BINARY : 임계값 이상이면 255, 아니면 0
        #THRESH_OTSU : 임계값을 자동으로 계산
        cv2.imshow('otsu_thread',otsu_thread)

 #### Morph-----------------------------------------------
        kernel =np.ones((5,5),np.uint8) #(5,5)적당한 정도
        erosion=cv2.erode(otsu_thread,kernel,iterations = 5)
        #침식 반대 다일라이트?, 축소(침식) 흰 노이즈 제거, 침식 5번 반복이라 효과 가 강함
        cv2.imshow('erosion',erosion)
        cv2.imwrite('digit_binary.png', erosion)

        #이미지 자르기-------------------------------------------------
        img = cv2.imread('digit_binary.png', cv2.IMREAD_UNCHANGED)
        h,w = img.shape[:2] #2행 가져오기
        crop_size =280 #가로 세로 이미지를 자름
        cx,cy = int(w/2),int(h/2)
        half = crop_size // 2
        x1,x2 =cx - half, cx + half
        y1,y2 = cy - half, cy + half

        #경계면 설정-----------------------------------------------
        x1= max(0,x1)
        y1= max(0,y1)
        x2= min(w,x2)
        y2= min(h,y2)

        cropped_img = img[y1:y2,x1:x2] #슬라이싱 범위이네
        cv2.imshow('cropped_img',cropped_img)

        #이미지 반전---------------------------------------------
        reversed_img = cv2.bitwise_not(cropped_img) #bit 반전
        cv2.imshow('reversed_img',reversed_img)
        cv2.imwrite('IMAG_FOR_TEST.png', reversed_img)

        #28*28이라 축소 해줘야한다

    if cv2.waitKey(1) ==27:
        break
cap.release()
cv2.destroyAllWindows()